In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

# Define the file path (joining the downloaded path with the filename)


file_path = os.path.join(path, "Q3_data.csv")

# 1. Read the dataset


df = pd.read_csv(file_path)


In [ ]:
# Task 2: Write your code here:
print("First 5 Rows ")
display(df.head())


In [ ]:
# Task 3: Write your code here:
# 3. Display dataset information (Check for nulls and data types)


print("\nDataset Information ")
df.info()

In [ ]:
# Task 4: Write your code here:
# 4. Show statistical description
print("\nStatistical Summary")
display(df.describe())

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Handle Missing Values
# For credit data, filling with median is often safer than mean to avoid outlier influence
df = df.fillna(df.median(numeric_only=True))
# For categorical columns, fill with "Unknown"
df = df.fillna("Na")


In [ ]:
# Task 2: Write your code here:

# 2. Check and remove duplicates
duplicate_count = df.duplicated().sum()



print(f"duplicate rows: {duplicate_count}")


if duplicate_count > 0:
    df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
# 3. Encode Categorical Variables
# Identify non-numeric columns (excluding the target 'target' if present)

cat_cols = df.select_dtypes(include=['object']).columns


le = LabelEncoder()


for col in cat_cols:


    df[col] = le.fit_transform(df[col].astype(str))


In [ ]:
# Task 4: Write your code here:
# 4. Apply Feature Scaling (StandardScaler)
# We scale features to have mean=0 and variance=1
scaler = StandardScaler()

features = df.drop(columns=['Target']) # Assuming 'target' is the label name


target = df['Target']

scaled_features = scaler.fit_transform(features)


df_scaled = pd.DataFrame(scaled_features, columns=features.columns)


In [ ]:
# Task 5: Write your code here:
# 5. Check for Target Imbalance
imbalance_ratio = target.value_counts(normalize=True) * 100


print("\nTarget Distribution")


print(imbalance_ratio)

if imbalance_ratio.min() < 20:

    print("\nResult: The dataset is IMBALANCED. We may need SMOTE or class weights later.")


else:


    print("\nResult: The dataset balance.")

In [ ]:
!pip install catboost

In [ ]:
# Task 1: Write your code here:

# 1. Split the dataset to features(X) and target(y)
X = df_scaled

# Using the scaled features from Part 2
y = target    # The target labels (0 or 1)


In [ ]:
# Task 2,3,4,5: Write your code here:

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# 2.StratifiedKFold
# 5 folds  robust validation


skf = StratifiedKFold(n_splits=5,
                      shuffle=True, random_state=42)

f1_scores = []



print("Cross-Validation")

# 3. Train CatBoost and Evaluate'

for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index],

    X.iloc[test_index]
    y_train, y_test = y.iloc[train_index],

    y.iloc[test_index]

    # Initialize CatBoost
    # We use silent=True to prevent the log from becoming too long


    model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, silent=True)

    # Train the model


    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)


    # 4. Evaluate using F1-Score
    score = f1_score(y_test, y_pred)


    f1_scores.append(score)


    print(f"Fold {fold + 1} F1-Score: {score:.4f}")

# 5. Print the averaged score

print("-" * 30)

print(f"Average F1-Score: {np.mean(f1_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Extract feature importance from the trained model


feature_importance = model.get_feature_importance()



feature_names = X.columns

# Create a DataFrame for easy sorting and plotting


data_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)



In [ ]:
# Task 2: Write your code here:
# 2. Plot Feature Importance
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=data_df.head(15), palette='magma')
plt.title('Top 15 Predictive Features (The Search for the Golden Feature)')
plt.xlabel('Importance Score')
plt.ylabel('Anonymized Feature Name')
plt.show()

# 3. Identify and print the 'Golden Feature'
golden_feature = data_df.iloc[0]['Feature']
golden_score = data_df.iloc[0]['Importance']

print(f"--- MISSION COMPLETE ---")
print(f"The Golden Feature is: {golden_feature}")
print(f"Importance Score: {golden_score:.2f}")

In [ ]:
# Task Bonus: Write your code here:
# 1. Create new X with ONLY the golden feature
# (Assuming 'golden_feature' was identified as 'P_2' or similar in Part 4)
X_golden = df_scaled[[golden_feature]]
y = target

# 2. Run the same KFold loop with this single feature
skf_solo = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
solo_f1_scores = []

print(f"Retraining model using ONLY: {golden_feature}...")

for train_index, test_index in skf_solo.split(X_golden, y):
    X_train_s, X_test_s = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train_s, y_test_s = y.iloc[train_index], y.iloc[test_index]

    # Using the same parameters as before
    solo_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, silent=True)
    solo_model.fit(X_train_s, y_train_s)

    # Predict and evaluate
    y_pred_s = solo_model.predict(X_test_s)
    solo_f1_scores.append(f1_score(y_test_s, y_pred_s))

# 3. Compare Results
avg_full_f1 = np.mean(f1_scores) # From Part 3
avg_solo_f1 = np.mean(solo_f1_scores)

print("\n" + "="*30)
print(f"Full Model F1-Score:    {avg_full_f1:.4f}")
print(f"Golden Feature Solo:    {avg_solo_f1:.4f}")
print(f"Performance Retained:   {(avg_solo_f1 / avg_full_f1) * 100:.2f}%")
print("="*30)
